In [2]:
import os

# Install PyPDF2
!pip install PyPDF2

import PyPDF2

# The user has provided a PDF file at this path:
pdf_path = '/content/sample.pdf.pdf'

if pdf_path:
    try:
        # Open the PDF file in binary mode
        with open(pdf_path, 'rb') as file:
            # Create a PdfReader object
            reader = PyPDF2.PdfReader(file)

            # Initialize an empty string to store all text
            all_text = ""

            # Loop through each page and extract text
            for page_num in range(len(reader.pages)):
                page = reader.pages[page_num]
                all_text += page.extract_text() + "\n"

            print("--- Extracted Text ---")
            print(all_text[:1000]) # Print first 1000 characters for brevity
            print(f"Total characters extracted: {len(all_text)}")

    except FileNotFoundError:
        print(f"Error: The file '{pdf_path}' was not found. Please upload the PDF or check the path.")
    except Exception as e:
        print(f"An error occurred while reading the PDF: {e}")
else:
    print("Skipping PDF processing as no valid PDF file was found or specified.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.3 MB/s eta 0:00:00
Error: The file '/content/sample.pdf.pdf' was not found. Please upload the PDF or check the path.


In [3]:
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits the input text into smaller, overlapping chunks.

    Args:
        text (str): The input text to be chunked.
        chunk_size (int): The desired size of each chunk.
        overlap (int): The number of characters to overlap between chunks.

    Returns:
        list: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start >= len(text) - overlap and start < len(text):
            # Ensure the last chunk includes the end of the text if it's too small for a full overlap
            if text[start:] not in chunks[-1]: # Avoid adding duplicate partial chunk
                chunks.append(text[start:])
            break

    # Remove any empty chunks or duplicate end chunks that might occur due to edge cases
    chunks = [chunk for chunk in chunks if chunk.strip()]

    return chunks

# Demonstrate the chunking function with the extracted text
# Let's use a chunk size of 500 characters with an overlap of 50 characters
text_chunks = chunk_text(all_text, chunk_size=500, overlap=50)

print(f"Original text length: {len(all_text)} characters")
print(f"Number of chunks created: {len(text_chunks)}")
print("\n--- First 3 Chunks ---")
for i, chunk in enumerate(text_chunks[:15]):
    print(f"Chunk {i+1} (length: {len(chunk)}):\n{chunk}\n")

# Store text_chunks for future use
# This will make the `text_chunks` variable available in the kernel for subsequent steps.


NameError: name 'all_text' is not defined

In [ ]:
# Install the sentence-transformers library
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained SentenceTransformer model
# 'all-MiniLM-L6-v2' is a good balance of performance and speed.
print("Loading SentenceTransformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully.")

# Generate embeddings for each text chunk
print("Generating embeddings for text chunks...")
text_embeddings = model.encode(text_chunks, show_progress_bar=True)
print(f"Embeddings generated for {len(text_embeddings)} chunks.")
print(f"Shape of one embedding: {text_embeddings[0].shape}")

# Display the first embedding for verification
print("\n--- First Embedding (first 5 values) ---")
print(text_embeddings[0][:5])

# Store text_embeddings for future use (e.g., storing in a vector database)


In [ ]:
# Install FAISS for efficient similarity search
!pip install faiss-cpu

In [ ]:
import faiss

# Ensure embeddings are in a contiguous float32 array, which FAISS expects
dimension = text_embeddings.shape[1]
embeddings_np = np.array(text_embeddings).astype('float32')

# Create a FAISS index
# IndexFlatL2 is a simple index that performs brute-force L2 distance search
print(f"Creating FAISS index with dimension: {dimension}")
index = faiss.IndexFlatL2(dimension)

# Add the embeddings to the index
print(f"Adding {len(embeddings_np)} embeddings to the FAISS index...")
index.add(embeddings_np)

print(f"FAISS index created and populated with {index.ntotal} vectors.")

# Store the index for future use


In [ ]:
# Function to search the FAISS index for relevant chunks
def search_faiss(query, model, index, text_chunks, k=5):
    """
    Embeds a query and searches the FAISS index for the top k most similar text chunks.

    Args:
        query (str): The search query.
        model (SentenceTransformer): The embedding model.
        index (faiss.Index): The FAISS index.
        text_chunks (list): The list of original text chunks.
        k (int): The number of top similar chunks to retrieve.

    Returns:
        list: A list of (score, text_chunk) tuples for the top k results.
    """
    # 1. Embed the query
    query_embedding = model.encode([query])

    # Ensure query embedding is float32 and 2D for FAISS search
    query_embedding_np = np.array(query_embedding).astype('float32')

    # 2. Perform similarity search
    # D: Distances (scores), I: Indices of the nearest neighbors
    D, I = index.search(query_embedding_np, k)

    # 3. Retrieve the actual text chunks
    results = []
    for i, idx in enumerate(I[0]):
        # FAISS returns L2 distance. Lower distance means higher similarity.
        # We can optionally convert to similarity score if needed, but for ranking, distance is fine.
        score = D[0][i]
        chunk = text_chunks[idx]
        results.append((score, chunk))

    # Sort by score (lower L2 distance is better)
    results.sort(key=lambda x: x[0])

    return results

# Example usage:
query = "What are Ayush Yadav's key skills and experience?"

print(f"\nSearching for: '{query}'")
search_results = search_faiss(query, model, index, text_chunks, k=3)

print("\n--- Top 3 Retrieved Chunks ---")
for i, (score, chunk) in enumerate(search_results):
    print(f"Rank {i+1} (L2 Distance: {score:.4f}):\n{chunk}\n")


## Integrating a Large Language Model (LLM) for Response Generation (using OpenRouter)

Now that we can retrieve relevant text chunks, the next step in building a RAG system is to use a Large Language Model (LLM) to synthesize a coherent answer based on these chunks and the user's query.

We will now use the **OpenRouter API** for this purpose. If you don't already have one, you'll need an API key from OpenRouter. Store your API key securely in Colab's secrets manager (the '🔑' icon in the left panel) under the name `OPENROUTER_API_KEY`.

In [ ]:
# Install the OpenAI client library, which can be used with OpenRouter's OpenAI-compatible API
!pip install -q -U openai

import openai
from google.colab import userdata

# Configure the OpenRouter API client
# Make sure to store your API key in Colab secrets under the name 'OPENROUTER_API_KEY'
try:
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    openai.api_key = OPENROUTER_API_KEY
    # OpenRouter's base URL for OpenAI-compatible API
    openai.base_url = "https://openrouter.ai/api/v1/"
    print("OpenRouter API configured successfully.")
except userdata.SecretNotFoundError:
    print("Error: OPENROUTER_API_KEY not found in Colab secrets.")
    print("Please add your OpenRouter API key to Colab secrets under the name 'OPENROUTER_API_KEY'.")
except Exception as e:
    print(f"An error occurred during OpenRouter API configuration: {e}")

In [ ]:
# Select an OpenRouter-compatible model
# Example models: 'mistralai/mistral-7b-instruct:free', 'google/gemini-pro'
# You can browse available models on openrouter.ai/docs#models
OPENROUTER_MODEL_NAME = 'mistralai/mistral-7b-instruct:free' # Using a free, popular option

print(f"LLM model '{OPENROUTER_MODEL_NAME}' selected for OpenRouter.")

## Integrating a Small Open-Source LLM Locally for Response Generation

To avoid external API dependencies and run an LLM directly within Colab, we can leverage models from Hugging Face's `transformers` library. We'll use a relatively small model like `microsoft/phi-2` which can run on Colab's free tier GPU. This approach eliminates the need for API keys for the LLM itself.

In [ ]:
# Install necessary libraries for loading Hugging Face models
!pip install -q transformers torch accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading local LLM model (microsoft/phi-2). This may take a few minutes...")

# Specify the model to use
LOCAL_LLM_MODEL_NAME = "microsoft/phi-2"

# Determine the device to use (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL_NAME, trust_remote_code=True)
model_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == torch.device('cuda') else torch.float32, # Use float16 for efficiency on GPU, float32 on CPU
    trust_remote_code=True
).to(device) # Move model to the determined device

print(f"Local LLM '{LOCAL_LLM_MODEL_NAME}' loaded successfully.")

### Generating a Response with the LLM (OpenRouter)

Now, let's create a function that takes the user's query and the retrieved context (our `search_results`) and feeds it to the LLM via OpenRouter to generate a human-like answer. We'll construct a prompt that clearly instructs the LLM to use only the provided context.

In [ ]:
def generate_rag_response(query, context_chunks, openrouter_model_name):
    """
    Generates a response using an LLM (via OpenRouter) based on a query and provided context chunks.

    Args:
        query (str): The user's query.
        context_chunks (list): A list of relevant text chunks (score, chunk) tuples.
        openrouter_model_name (str): The name of the OpenRouter model to use.

    Returns:
        str: The generated answer from the LLM.
    """

    # Combine the context chunks into a single string
    context = "\n\n".join([chunk for score, chunk in context_chunks])

    # Construct the prompt for the LLM
    # OpenRouter expects messages in the OpenAI chat format
    messages = [
        {"role": "system", "content": "You are an intelligent assistant designed to provide answers based *only* on the context provided. If the answer cannot be found in the context, state that you don't have enough information. Do not make up information."},
        {"role": "user", "content": f"Query: {query}\n\nContext:\n{context}\n\nAnswer:"}
    ]

    try:
        response = openai.chat.completions.create(
            model=openrouter_model_name,
            messages=messages,
            temperature=0.7 # Adjust temperature for creativity vs. factualness
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating response from OpenRouter: {e}"

# Generate the response using the retrieved chunks
# The 'query' and 'search_results' variables should already be defined from previous steps
rag_answer = generate_rag_response(query, search_results, OPENROUTER_MODEL_NAME)

print("\n--- Generated RAG Answer (via OpenRouter) ---")
print(rag_answer)

In [ ]:
# Ensure FAISS, SentenceTransformers, and Hugging Face transformers are installed
!pip install -q faiss-cpu sentence-transformers transformers accelerate

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Re-create RAG pipeline components for this cell's scope ---
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Splits the input text into smaller, overlapping chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
        if start >= len(text) - overlap and start < len(text):
            if text[start:] not in chunks[-1]:
                chunks.append(text[start:])
            break
    chunks = [chunk for chunk in chunks if chunk.strip()]
    return chunks

# Safely retrieve the extracted text
if 'all_text' in globals() and all_text.strip():
    print("Using successfully extracted text from the PDF.")
else:
    print("Warning: 'all_text' not found in globals. Setting fallback placeholder.")
    all_text = "Ayush Yadav. Cost Control Engineer with over 4+ years of experience in cost management, project controls, budgeting, forecasting within Western Bainoona Group, Hard Precast Building Systems, and Workz Group. Education: Amity University (B.Tech, Mechanical Engineering)."

# Re-create text_chunks for this cell's scope
text_chunks = chunk_text(all_text, chunk_size=500, overlap=50)

# Re-load SentenceTransformer model specifically for embedding
model_sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')

# Re-generate embeddings for text_chunks
text_embeddings = model_sentence_transformer.encode(text_chunks, show_progress_bar=False)
dimension = text_embeddings.shape[1]
embeddings_np = np.array(text_embeddings).astype('float32')

# Re-create FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_np)

# --- Load Local LLM (microsoft/phi-2) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device for LLM: {device}")

LOCAL_LLM_MODEL_NAME = "microsoft/phi-2"
print(f"Loading local LLM '{LOCAL_LLM_MODEL_NAME}'. This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL_NAME, trust_remote_code=True)
model_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == torch.device('cuda') else torch.float32,
    trust_remote_code=True
).to(device)
print(f"Local LLM '{LOCAL_LLM_MODEL_NAME}' and tokenizer loaded successfully.")
# -------------------------------------------------------------------------

def generate_rag_response_local(query, context_chunks, tokenizer, model_llm):
    """
    Generates a response using a locally loaded LLM (via Hugging Face transformers)
    based on a query and provided context chunks.
    """
    context = "\n\n".join([chunk for score, chunk in context_chunks])

    prompt = (
        f"Instruct: Based on the following context, answer the query. If the answer cannot be found in the context, state that you don't have enough information. Do not make up information.\n\n"
        f"Context:\n{context}\n\n"
        f"Query: {query}\n\n"
        f"Output:"
    )

    try:
        device_llm = model_llm.device
        inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=False).to(device_llm)

        outputs = model_llm.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.3, top_p=0.9)
        text_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "Output:" in text_output:
            generated_answer = text_output.split("Output:", 1)[1].strip()
        else:
            generated_answer = text_output.strip()

        return generated_answer
    except Exception as e:
        return f"Error generating response from local LLM: {e}"


def search_faiss(query, model_sentence_transformer, index, text_chunks, k=5):
    query_embedding = model_sentence_transformer.encode([query])
    query_embedding_np = np.array(query_embedding).astype('float32')
    D, I = index.search(query_embedding_np, k)

    results = []
    for i, idx in enumerate(I[0]):
        score = D[0][i]
        chunk = text_chunks[idx]
        results.append((score, chunk))

    results.sort(key=lambda x: x[0])
    return results

# Define query
query = "What are Ayush Yadav's key skills and experience?"

# Run search_faiss
search_results = search_faiss(query, model_sentence_transformer, index, text_chunks, k=3)

# Call local RAG
rag_answer_local = generate_rag_response_local(query, search_results, tokenizer, model_llm)

print("\n--- Generated RAG Answer (via Local LLM) ---")
print(rag_answer_local)

In [ ]:
def get_complex_prompt_template(system_role, context, query, instructions=None):
    """
    Creates a structured and robust prompt template for complex QA generation.
    """
    default_instructions = (
        "1. Rely only on clear facts directly mentioned in the context.\n"
        "2. Do not assume, extrapolate, or bring in outside knowledge.\n"
        "3. If the answer cannot be fully found in the context, clearly outline what is missing.\n"
        "4. Format your response clearly using bullet points where appropriate."
    )

    active_instructions = instructions if instructions else default_instructions

    prompt = (
        f"System: {system_role}\n\n"
        f"Context:\n{context}\n\n"
        f"Instructions:\n{active_instructions}\n\n"
        f"User Query: {query}\n\n"
        f"Output:"
    )
    return prompt

# Example Usage:
system_role = "You are a professional recruiting assistant evaluating candidate documents for specific job fits."
context = "\n\n".join([chunk for score, chunk in search_results])
query = "Summarize Ayush's professional background and highlight his technical software proficiencies."

complex_prompt = get_complex_prompt_template(system_role, context, query)
print("--- Formatted Complex Prompt ---")
print(complex_prompt)

In [ ]:
try:
    # Move inputs to the same device as the model
    device_llm = model_llm.device
    inputs = tokenizer(complex_prompt, return_tensors="pt", return_attention_mask=False).to(device_llm)

    # Generate output with strict guidelines
    print("Generating complex answer with local LLM...")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.2,
        top_p=0.9
    )
    text_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Display output
    if "Output:" in text_output:
        final_response = text_output.split("Output:", 1)[1].strip()
    else:
        final_response = text_output.strip()

    print("\n--- Generated Response ---")
    print(final_response)
except Exception as e:
    print(f"An error occurred during generation: {e}")

In [ ]:
# 1. Define a completely different persona and query for the test case
technical_interviewer_persona = "You are a strict, highly technical engineering manager and interviewer looking for concrete mathematical or physical project evidence."

context_for_test = "\n\n".join([chunk for score, chunk in search_results])

test_query = "Identify any hardware or physical engineering projects Ayush has built, detailing their specifications or mechanisms if mentioned."

# 2. Format the new prompt using our template function
test_complex_prompt = get_complex_prompt_template(
    system_role=technical_interviewer_persona,
    context=context_for_test,
    query=test_query
)

# 3. Generate response using the local model
try:
    device_llm = model_llm.device
    inputs = tokenizer(test_complex_prompt, return_tensors="pt", return_attention_mask=False).to(device_llm)

    print("Generating technical evaluation response...")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.2,
        top_p=0.9
    )
    text_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Output:" in text_output:
        test_response = text_output.split("Output:", 1)[1].strip()
    else:
        test_response = text_output.strip()

    print("\n--- Technical Interviewer Response ---")
    print(test_response)
except Exception as e:
    print(f"An error occurred during evaluation: {e}")